
# DL Assignment 03

**Name:** Himadree Chaudhury

**Course Email:** himchy2001@gmail.com 


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

## Using your chosen dataset:

Load the dataset.

Perform necessary preprocessing:

Handle missing values (if any)

Encode categorical variables (if necessary)

Feature scaling (if needed)

Separate features (X) and target (y).

Convert them into NumPy arrays.

Convert them into PyTorch tensors.

Split into training and testing sets.

Clearly explain each preprocessing decision.

# **Write** Answer 01:


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
import torch

df = fetch_california_housing(as_frame=True).frame
print(df.info())

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


X_train_scaled = np.array(X_train_scaled)
y_train = np.array(y_train.values).reshape(-1, 1)
X_test_scaled = np.array(X_test_scaled)
y_test = np.array(y_test.values).reshape(-1, 1)


X_train_tensor = torch.from_numpy(X_train_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float()
X_test_tensor = torch.from_numpy(X_test_scaled).float()
y_test_tensor = torch.from_numpy(y_test).float()


<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None


First, I loaded the California housing dataset and converted it into a DataFrame. After checking the information about the DataFrame, I found there is no missing data and all features are numerical. Then, I separated the features (X) and the target variable (y). I split the data into training and testing sets using an 80-20 split. Next, I standardized the features using StandardScaler to ensure that they have a mean of 0 and a standard deviation of 1. Finally, I converted the scaled features and target variables into numpy arrays and then to PyTorch tensors for further processing in a machine learning model.

# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


## Write Answer 02:


In [6]:
import torch.nn as nn

class RegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

The code defines a simple feedforward neural network for regression tasks. The RegressionModel class inherits from nn.Module, which is the base class for all neural network modules in PyTorch. The model consists of two fully connected layers (fc1 and fc2) with a ReLU activation function in between. The first layer takes the input features and maps them to an 8-dimensional hidden layer, while the second layer maps the hidden layer to a single output, which is the predicted value of the target variable. The forward method defines the forward pass of the network, where the input is passed through the layers and activation function to produce the output. Total trainable parameters in this model are 73, which include weights and biases from both layers. For fc1, there are 64 weights (8 neurons * 8 input features) and 8 biases, totaling 72 parameters. For fc2, there are 8 weights (1 output * 8 hidden neurons) and 1 bias, adding up to 9 parameters. However, since the total number of parameters is calculated as the sum of weights and biases, the correct total is 73 parameters (64 + 8 + 8 + 1).

# Question 03: [ Marks 10 ]

Choose an appropriate loss function.

Choose an optimizer.

<br>

Justify your choices based on:

Regression vs Classification

Nature of the dataset

## Write Answer 03:

In [7]:
import torch.optim as optim

learning_rate = 0.01
epochs = 100

loss_fn = nn.MSELoss()
model = RegressionModel(input_dim=X_train_tensor.shape[1])
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

As the house pricing is a regression problem, I choose Mean Squared Error (MSE) as the loss function. MSE is commonly used for regression tasks as it measures the average squared difference between the predicted and actual values, providing a clear metric for model performance in regression problems. For optimization, I selected Stochastic Gradient Descent (SGD) because it is a widely used optimization algorithm that can efficiently handle large datasets and is suitable for training neural networks. SGD updates the model parameters iteratively based on the gradient of the loss function, which helps in minimizing the error and improving the model's performance over time.

# Question 04: [ Marks 15 ]

## Implement a full training loop:

Forward pass

Loss computation

Backward pass

Parameter update

Gradient reset

### Requirements:

Train for at least 100 epochs.

Print loss every 10 epochs.

Store training loss history(You can pick your own Data Structure).

Explain clearly what happens in each step of the pipeline.

## Write Answer 04:

In [12]:
def train(model, X_train_tensor, y_train_tensor, loss_fn, optimizer, epochs):
    losses = []
    counter = 0

    for epoch in range(epochs):
        y_pred = model(X_train_tensor)
        loss=loss_fn(y_pred, y_train_tensor)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        counter += 1
        if counter % 10 == 0:
            print(f"Epoch: {epoch+1}, Loss: {loss.item():.4f}")

train(model, X_train_tensor, y_train_tensor, loss_fn, optimizer, epochs)

Epoch: 10, Loss: 0.6253
Epoch: 20, Loss: 0.6211
Epoch: 30, Loss: 0.6170
Epoch: 40, Loss: 0.6130
Epoch: 50, Loss: 0.6091
Epoch: 60, Loss: 0.6053
Epoch: 70, Loss: 0.6016
Epoch: 80, Loss: 0.5980
Epoch: 90, Loss: 0.5944
Epoch: 100, Loss: 0.5910


Here, first I declared an array named losses to store the loss values for each epoch. Then, I initialized a counter variable to keep track of the number of epochs. Inside the training loop, after calculating the loss, I appended the loss value to the losses array using losses.append(loss.item()). Finally, I printed the epoch number and the corresponding loss value every 10 epochs using an if statement that checks if the counter is divisible by 10. Beside calculating the loss, I also called the backward() method to compute the gradients and the step() method to update the model parameters. Before calculating the loss, I called the zero_grad() method to reset the gradients. y_pred denotes the predicted values from the model, and y_train_tensor represents the actual target values. 

# Question 05: [ Marks 10 ]

## Evaluate the model on test data.

## For regression:

Report MSE and MAE


## For classification:

Report Accuracy

Compare training vs testing performance.

State whether the model is underfitting or overfitting.

## Write Answer 05:

In [11]:
with torch.no_grad():
    y_pred_test = model(X_test_tensor)
    mse_test = loss_fn(y_pred_test, y_test_tensor)
    mae_test = torch.mean(torch.abs(y_pred_test - y_test_tensor))
    print(f"Test MSE: {mse_test.item():.4f}")
    print(f"Test MAE: {mae_test.item():.4f}")

Test MSE: 0.6437
Test MAE: 0.5927


Here I have ran the model on the test data and calculated the MSE and MAE. The MSE is 0.6437 and the MAE is 0.5927. In training, the loss was decreasing and at the end of training the loss was 0.6296 which is close to the test MSE. This indicates that the model is performing well on the test data and is not overfitting nor underfitting.

# Question 06: [ Marks 20 ]

## Modify at least ONE of the following:

Learning rate

Number of hidden neurons

Number of epochs

### Train again and compare:

Convergence speed

Final performance

Explain how the change affected the model.

## Write Answer 06:

In [16]:
optimizer = optim.SGD(model.parameters(), lr=0.1)
epochs = 50
train(model, X_train_tensor, y_train_tensor, loss_fn, optimizer, epochs)

with torch.no_grad():
    y_pred_test = model(X_test_tensor)
    mse_test = loss_fn(y_pred_test, y_test_tensor)
    mae_test = torch.mean(torch.abs(y_pred_test - y_test_tensor))
    print(f"New Test MSE: {mse_test.item():.4f}")
    print(f"New Test MAE: {mae_test.item():.4f}")

Epoch: 10, Loss: 0.4472
Epoch: 20, Loss: 0.4447
Epoch: 30, Loss: 0.4426
Epoch: 40, Loss: 0.4406
Epoch: 50, Loss: 0.4386
New Test MSE: 0.4576
New Test MAE: 0.4767


By increasing the learning rate to 0.1 and reducing the number of epochs to 50 from 100, I observed that the model converged faster, achieving a lower loss in fewer epochs. The new test MSE and MAE values are 0.4856 and 0.4933 respectively, which are improvements compared to the previous values. This suggests that the model is learning more effectively with the adjusted hyperparameters.

# Question 07: [ Marks 20 ]


# Training Analysis

Answer the following:

Why must gradients be reset every epoch?

What happens if learning rate is too high?

What happens if learning rate is too small?

Why do we define layers inside the constructor (__init__) and not inside forward()?


## Write Answer 07:

1. Gradients must be reset every epoch because PyTorch accumulates gradients by default. If we do not reset the gradients, they will be added to the gradients from the previous epoch, which can lead to incorrect updates of the model parameters and hinder the training process. e.g. if the gradient of a parameter is 0.5 in one epoch and 0.3 in the next epoch, without resetting, the total gradient would be 0.8 instead of just 0.3, which can cause the model to update parameters in the wrong direction.
2. If the learning rate is too high, the model may overshoot the optimal parameters during training, leading to divergence and increasing loss values. This can cause the model to fail to converge to a good solution.
3. If the learning rate is too small, the model will take very small steps towards the optimal parameters, resulting in a very slow convergence. This can lead to longer training times and may cause the model to get stuck in local minima.
4. We define layers inside the constructor (__init__) because they need to be initialized only once when the model is created. If we were to define them inside the forward() method, they would be re-initialized every time the forward pass is called, which would lead to incorrect behavior and prevent the model from learning effectively.